# Пара 12. Бесплатные API под **вашего** автора + проверка цитат

Gutendex, Open Library, Wikidata — слой каталога/фактов рядом с текстом диссертации.  
Плюс мини-протокол проверки ответа LLM (если используете чат для идей).


## Старт в Colab
1. **Файл → Сохранить копию на Диске** (чтобы правки не пропали).
2. Справа сверху — **Подключить**.
3. Ячейки сверху вниз: Play или Shift+Enter.
Базовый Colab **бесплатный**. Правило занятия: **вопрос → процедура → картинка → цитата → ограничение**.


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

AUTHOR = "Ivan Bunin"  # замените: автор/фигура вашей диссертации
AUTHOR_QID = "Q233005"  # Wikidata Q-id автора (найдите на wikidata.org)

# 1) Gutendex
g = requests.get("https://gutendex.com/books", params={"search": AUTHOR}, timeout=60).json()
print("Gutendex count", g.get("count"))

# 2) Open Library
ol = requests.get(
    "https://openlibrary.org/search.json",
    params={"author": AUTHOR, "limit": 50},
    timeout=60,
).json()
print("Open Library numFound", ol.get("numFound"))
years = [d.get("first_publish_year") for d in ol.get("docs", []) if d.get("first_publish_year")]
pd.Series(years).value_counts().sort_index().plot(kind="bar", figsize=(10, 3), title=f"OL years · {AUTHOR}")
plt.tight_layout(); plt.show()


In [ ]:
# 3) Wikidata SPARQL — связанные сущности
q = f'''SELECT ?item ?itemLabel WHERE {{
  ?item ?p wd:{AUTHOR_QID} .
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "ru,en". }}
}} LIMIT 30'''
r = requests.get(
    "https://query.wikidata.org/sparql",
    params={"query": q, "format": "json"},
    headers={"User-Agent": "SamGU-DH-Course/1.0"},
    timeout=60,
)
bindings = r.json()["results"]["bindings"]
print(len(bindings), "rows")
for b in bindings[:12]:
    print(b.get("itemLabel", {}).get("value"))


## Протокол LLM (если берёте идеи у чата)
Скопируйте в Markdown-ячейку и заполните:

1. Фрагмент текста (не больше страницы) — вставлен мной.  
2. Промпт (роль / задача / формат с полями цитата+уверенность).  
3. Ответ модели.  
4. **Сверка по изданию:** какие цитаты совпали / какие галлюцинация.  
5. Что переносится в диссертацию только после сверки.

## Сдать
1 график OL или таблица Wikidata + заполненный протокол LLM **или** абзац, почему LLM на этом шаге не нужен.
